# Determining UFC Fight Outcomes Using a Siamese Neural Network

**Author:** [Your Name]  
**Affiliation:** [University / Lab / Organization]  
**Email:** [your.email@example.com]  
**Date:** [Month DD, YYYY]


## Abstract

Predicting mixed martial arts outcomes is challenging due to stylistic matchups, sparse direct head-to-head data, and fighter performance drift over time. This paper presents a Siamese neural network architecture for UFC fight outcome prediction by learning shared fighter representations from sequential fight history and combining them through a pairwise decision head. We evaluate the proposed model against tabular baselines (naive pre-fight win-rate, logistic regression, gradient boosting, and MLP) under a leakage-safe temporal split. Evaluation includes ROC-AUC, log loss, accuracy, Brier score, expected calibration error (ECE), subgroup analysis, and ablation studies. The full implementation is provided in reproducible scripts and notebook workflows for direct replication and extension.

**Keywords:** UFC, Siamese network, fight prediction, RNN, temporal validation, sports analytics


## 1. Introduction

UFC outcome prediction is a sequential and relational problem. Each bout is influenced by:

1. Fighter-specific evolving form (wins/losses, rest, historical efficiency).
2. Matchup asymmetry (style and physical differences).
3. Context (weight class, title status, rounds).

A Siamese architecture is a natural fit because both fighters can be encoded with a shared temporal module, then compared via pairwise interactions.


### 1.1 Research Questions

1. Does a Siamese RNN outperform strong tabular baselines under temporal evaluation?
2. Which feature groups matter most (context, efficiency, interaction design)?
3. Is model confidence calibrated and stable across relevant subgroups?


### 1.2 Contributions

1. End-to-end UFC fight prediction pipeline with leakage-safe chronological processing.
2. Shared-encoder Siamese GRU design for pairwise fight classification.
3. Baseline benchmarking, calibration analysis, subgroup reporting, and ablations.
4. Fully reproducible code implementation.


## 2. Related Work

Prior work generally falls into:

1. Sports and betting prediction using static classifiers.
2. MMA-specific feature engineering (record summaries, striking and grappling rates).
3. Pairwise and metric-learning methods where two entities are jointly modeled.
4. Sequence models for non-stationary behavior over time.

This study differs by combining chronological fighter histories with a Siamese RNN and explicit anti-leakage temporal splitting.


## 3. Data


### 3.1 Data Sources

The dataset is scraped from UFCStats and stored in a single fight-level CSV:

- `data/ufc_fights_rnn.csv`
- generated by `scripts/scrape_ufc_fights.py`


### 3.2 Inclusion / Exclusion Criteria

1. Included: completed UFC fights with binary outcomes.
2. Excluded from training target: draws/no-contests/unknown outcomes.
3. Dropped rows with invalid event dates.


### 3.3 Target Definition

- `y = 1` if `outcome_label == fighter_1_win`
- `y = 0` if `outcome_label == fighter_2_win`


### 3.4 Feature Engineering

Features are grouped as:

1. Fighter bio and demographics (age, height, reach, stance).
2. Pre-fight record/form (wins/losses/draws/NC, streak, layoff).
3. Pre-fight efficiency aggregates (striking, takedown, sub attempts, control).
4. Method splits (KO/Sub/Decision pre-fight records).
5. Matchup deltas (age/reach/experience/win-rate/rest differences).
6. Context (weight class, title bout, rounds, bout index, location).


### 3.5 Leakage Prevention

All pre-fight features are computed chronologically and only use prior history.

```python
# scripts/scrape_ufc_fights.py
row = build_fight_row(..., state_1=state_1, state_2=state_2)  # pre-fight state
store.insert_fight(row)
store.upsert_fighter_state(fighter_1_id, apply_result_to_state(...))  # update after insert
store.upsert_fighter_state(fighter_2_id, apply_result_to_state(...))
```


### 3.6 Data Collection Snippet

```bash
python3 scripts/scrape_ufc_fights.py \
  --max-events 50 \
  --commit-every 50 \
  --sleep-seconds 0.0
```


## 4. Methodology


### 4.1 Problem Formulation

Given fighters \(A\) and \(B\), predict:

\[
P(A \text{ beats } B \mid x_A^{pre}, x_B^{pre}, c)
\]

where \(x_A^{pre}, x_B^{pre}\) are pre-fight fighter histories/features and \(c\) is fight context.


### 4.2 Siamese Architecture

Each fighter sequence is passed through a shared GRU encoder:

```python
# scripts/run_ufc_siamese_study.py
h1 = self.encode_sequence(seq1, len1)
h2 = self.encode_sequence(seq2, len2)
pair = torch.cat([h1, h2, torch.abs(h1 - h2), h1 * h2, s], dim=1)
logits = self.head(pair).squeeze(1)
```


### 4.3 Temporal Sequence Construction

Fighter histories are built in event order, and each sample uses only prior fights:

```python
# scripts/run_ufc_siamese_study.py
seq1_hist = histories.get(fighter_1_id, [])
seq2_hist = histories.get(fighter_2_id, [])
samples.append({"seq1_hist": list(seq1_hist[-max_seq_len:]), ...})
histories.setdefault(fighter_1_id, []).append(f1_entry)  # append current after snapshot
histories.setdefault(fighter_2_id, []).append(f2_entry)
```


### 4.4 Baselines

Implemented baselines:

1. Naive pre-fight win-rate.
2. Logistic Regression.
3. HistGradientBoostingClassifier.
4. MLPClassifier.

```python
# scripts/run_ufc_siamese_study.py
models = ["logistic_regression", "gradient_boosting", "mlp"]
naive_probs = naive_prefight_winrate_probs(test_df)
```


## 5. Experimental Setup


### 5.1 Temporal Split

Chronological split by event date (train/val/test):

```python
# scripts/run_ufc_siamese_study.py
split = np.where(
    df["event_date"].dt.normalize() <= train_end, "train",
    np.where(df["event_date"].dt.normalize() <= val_end, "val", "test"),
)
df["__split__"] = split
```


### 5.2 Metrics

Primary:

1. ROC-AUC
2. Log loss

Secondary:

1. Accuracy
2. Brier score
3. ECE

```python
# scripts/run_ufc_siamese_study.py
return {
    "roc_auc": roc_auc_score(y, p),
    "log_loss": log_loss(y, p),
    "accuracy": accuracy_score(y, preds),
    "brier": brier_score_loss(y, p),
    "ece": expected_calibration_error(y, p, n_bins=10),
}
```


### 5.3 Training Setup

```python
# scripts/run_ufc_siamese_study.py
siamese_cfg = SiameseConfig(
    max_seq_len=8, hidden_dim=96, batch_size=64,
    epochs=25, lr=1e-3, weight_decay=1e-4, patience=5
)
```


### 5.4 Optimization and Early Stopping

```python
# scripts/run_ufc_siamese_study.py
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
criterion = nn.BCEWithLogitsLoss()
if val_metrics["log_loss"] < best_val_score:
    best_state = copy.deepcopy(model.state_dict())
else:
    no_improve_epochs += 1
    if no_improve_epochs >= config.patience:
        break
```


## 6. Results


### 6.1 Main Comparison Table

Results are exported automatically:

- `outputs/<run_name>/main_metrics.csv`

```python
import pandas as pd
metrics = pd.read_csv("outputs/paper_run_01/main_metrics.csv")
metrics.sort_values("test_log_loss")
```


### 6.2 Calibration

```python
# scripts/run_ufc_siamese_study.py
calibration_df = calibration_table(y_test, probs, n_bins=10)
calibration_df.to_csv(out_dir / f"calibration_{model_name}.csv", index=False)
```


### 6.3 Subgroup Analysis

```python
# scripts/run_ufc_siamese_study.py
subgroup_df = subgroup_metrics(test_df_local, probs, model_name)
subgroup_df.to_csv(out_dir / f"subgroup_{model_name}.csv", index=False)
```


### 6.4 Ablation Study

Implemented ablations:

1. `siamese_no_context`
2. `siamese_no_efficiency`
3. `siamese_concat_only`

```python
# scripts/run_ufc_siamese_study.py
variants = [
    ("siamese_no_context", "full", [], [...]),
    ("siamese_no_efficiency", "full", ["sig_str", "td_", "sub_", "control", "knockdowns"], []),
    ("siamese_concat_only", "concat", [], []),
]
```


## 7. Discussion

Observed behavior should be interpreted across:

1. Discrimination vs calibration tradeoff.
2. Sequence contribution beyond static tabular features.
3. Weight-class and context-dependent performance variation.
4. Practical uncertainty for close matchup probabilities.


## 8. Limitations

1. Public MMA data quality and schema drift.
2. Missing latent variables (camp changes, injuries, weight cut quality).
3. Non-stationarity over long eras.
4. Potential sample imbalance by era/division.


## 9. Ethical and Practical Considerations

1. Avoid overconfident deployment in betting contexts.
2. Report calibration and uncertainty, not only accuracy.
3. Keep reproducibility artifacts and versioned preprocessing.


## 10. Conclusion

This work implements and operationalizes a full Siamese RNN UFC prediction study with temporally valid evaluation and reproducible outputs. The framework supports baseline comparison, calibration analysis, subgroup diagnostics, and ablations, enabling defensible empirical conclusions for pairwise fight modeling.


## References

Add full references for:

1. Siamese and metric learning architectures.
2. RNN/GRU sequence modeling.
3. Calibration metrics and reliability analysis.
4. MMA/UFC prediction literature.
5. Tooling references (PyTorch, scikit-learn, UFCStats data source).


## Appendix


### A. Reproducibility Checklist

1. Seed control (`--seed`).
2. Temporal split boundaries stored in `run_metadata.json`.
3. Model hyperparameters saved in `run_metadata.json`.
4. Prediction outputs saved in `test_predictions.csv`.
5. Training traces saved in `siamese_training_history.csv`.


### B. End-to-End Commands

```bash
# 1) Build dataset
python3 scripts/scrape_ufc_fights.py --max-events 100 --commit-every 20

# 2) Run full study
python3 scripts/run_ufc_siamese_study.py \
  --input-csv data/ufc_fights_rnn.csv \
  --output-dir outputs \
  --run-name paper_run_01

# 3) Open notebook workbench
# ufc_siamese_experiment_workbench.ipynb
```
